<a href="https://colab.research.google.com/github/hami913/AI-Credit-Default-Prediction-System/blob/colab-notebook/Credit_default_pridiction_model_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORT LIBARARIES

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

LOAD DATASET

In [ ]:
df=pd.read_csv("/content/UCI_Credit_Card.csv")
df.head()

DATSET PREPROCESSING

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:

missing_values = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df)) * 100
})

print("Total missing values:", df.isnull().sum().sum())

display(
    missing_values.sort_values(
        by="Missing_Count",
        ascending=False
    )
)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Total duplicate rows:", duplicate_count)

if duplicate_count > 0:
    print("\nDuplicate rows:")
    display(df[df.duplicated(keep=False)].sort_values("ID"))
else:
    print("No completely duplicated rows found.")

In [ ]:
print("Total rows:", len(df))
print("Unique IDs:", df["ID"].nunique())
print("Duplicate IDs:", df["ID"].duplicated().sum())

print("\nMinimum ID:", df["ID"].min())
print("Maximum ID:", df["ID"].max())

In [ ]:
categorical_cols = ["SEX", "EDUCATION", "MARRIAGE"]

for col in categorical_cols:
    print(f"\n{'=' * 40}")
    print(f"{col} - Unique values: {sorted(df[col].unique())}")
    print(f"{'=' * 40}")

    print(df[col].value_counts().sort_index())

In [ ]:
target = "default.payment.next.month"

target_counts = df[target].value_counts().sort_index()
target_percent = df[target].value_counts(normalize=True).sort_index() * 100

target_distribution = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percent
})

target_distribution.index = ["No Default (0)", "Default (1)"]

display(target_distribution)

print(f"Default rate: {target_percent[1]:.2f}%")
print(f"Non-default rate: {target_percent[0]:.2f}%")
print(f"Imbalance ratio: {target_counts[0] / target_counts[1]:.2f} : 1")

Check unique values and frequencies in repayment status columns

In [ ]:
pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

for col in pay_cols:
    print(f"\n{'=' * 45}")
    print(f"{col} - Unique values: {sorted(df[col].unique())}")
    print(f"{'=' * 45}")

    print(df[col].value_counts().sort_index())

Create a working copy before cleaning

In [ ]:
df_clean = df.copy()
df_clean["EDUCATION"] = df_clean["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})
df_clean["MARRIAGE"] = df_clean["MARRIAGE"].replace({
    0: 3
})

print("EDUCATION after cleaning:")
print(df_clean["EDUCATION"].value_counts().sort_index())

print("\nMARRIAGE after cleaning:")
print(df_clean["MARRIAGE"].value_counts().sort_index())

print("\nOriginal dataset shape:", df.shape)
print("Clean dataset shape:", df_clean.shape)

 Check ranges of important numerical features

In [ ]:
numerical_cols = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

range_summary = pd.DataFrame({
    "Minimum": df_clean[numerical_cols].min(),
    "Maximum": df_clean[numerical_cols].max(),
    "Mean": df_clean[numerical_cols].mean(),
    "Median": df_clean[numerical_cols].median()
})

display(range_summary)

Check skewness of numerical features

In [ ]:
skewness = df_clean[numerical_cols].skew().sort_values(ascending=False)

skewness_table = pd.DataFrame({
    "Skewness": skewness
})

display(skewness_table)

Correlation of all features with target

In [ ]:
target = "default.payment.next.month"

correlation_with_target = (
    df_clean.corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(ascending=False)
)

correlation_table = pd.DataFrame({
    "Correlation_with_Default": correlation_with_target
})

display(correlation_table)

Check highly correlated feature pairs

In [ ]:
 feature_df = df_clean.drop(
    columns=["ID", "default.payment.next.month"]
)

corr_matrix = feature_df.corr(numeric_only=True).abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
)

high_corr_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Correlation"] > 0.70
].sort_values(
    "Correlation",
    ascending=False
)

display(high_corr_pairs)

Define target

In [ ]:
target = "default.payment.next.month"
df_clean = df_clean.dropna(subset=[target]).copy()
X = df_clean.drop(columns=["ID", target])
y = df_clean[target]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Original dataset after removing missing target rows:", df_clean.shape)

print("\nFeature/Target shapes:")
print("X:", X.shape)
print("y:", y.shape)
print("\nTrain/Test shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("\nClass distribution:")
print(f"Full  default rate: {y.mean() * 100:.2f}%")
print(f"Train default rate: {y_train.mean() * 100:.2f}%")
print(f"Test  default rate: {y_test.mean() * 100:.2f}%")

Define feature groups

In [ ]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)
print("\nNumber of categorical features:", len(categorical_features))
print("\nNumerical/ordinal features:")
print(numerical_features)
print("\nNumber of numerical/ordinal features:", len(numerical_features))
print("\nTotal features:",
      len(categorical_features) + len(numerical_features))

PIPELINE WORKING...... Numerical preprocessing

In [ ]:
numerical_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)
print("Preprocessor created successfully!")
print(preprocessor)

Fit preprocessor ONLY on training data

In [ ]:
 X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Preprocessing completed successfully!")
print("\nBefore preprocessing:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\nAfter preprocessing:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)
print("\nData type:")
print(type(X_train_processed))

Get final feature names after preprocessing

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Total final features:", len(feature_names))
print("\nFinal feature names:")
for i, feature in enumerate(feature_names, start=1):
    print(f"{i}. {feature}")
print("\nNaN values in processed training data:",
      np.isnan(X_train_processed).sum())
print("NaN values in processed test data:",
      np.isnan(X_test_processed).sum())
print("Infinite values in processed training data:",
      np.isinf(X_train_processed).sum())
print("Infinite values in processed test data:",
      np.isinf(X_test_processed).sum())

Create baseline Logistic Regression model

In [ ]:
import time
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
start_time = time.time()
logistic_model.fit(X_train_processed, y_train)
training_time = time.time() - start_time
print("Logistic Regression training completed!")
print(f"Training time: {training_time:.2f} seconds")
print("\nModel parameters:")
print("Solver:", logistic_model.solver)
print("Penalty:", logistic_model.penalty)
print("C:", logistic_model.C)
print("Class weight:", logistic_model.class_weight)
print("Iterations used:", logistic_model.n_iter_[0])

 Predictions

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


y_pred = logistic_model.predict(X_test_processed)


y_prob = logistic_model.predict_proba(X_test_processed)[:, 1]


accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("Baseline Logistic Regression Results")
print("=" * 45)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

 Logistic Regression with balanced class weights

In [ ]:
 import time
logistic_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

start_time = time.time()

logistic_balanced.fit(
    X_train_processed,
    y_train
)

training_time_balanced = time.time() - start_time

print("Balanced Logistic Regression training completed!")
print(f"Training time: {training_time_balanced:.2f} seconds")

print("\nModel parameters:")
print("Solver:", logistic_balanced.solver)
print("Penalty:", logistic_balanced.penalty)
print("C:", logistic_balanced.C)
print("Class weight:", logistic_balanced.class_weight)
print("Iterations used:", logistic_balanced.n_iter_[0])

PRIDICTIOS......EVALUATION

In [ ]:

y_pred_balanced = logistic_balanced.predict(X_test_processed)
y_prob_balanced = logistic_balanced.predict_proba(X_test_processed)[:, 1]
accuracy_bal = accuracy_score(y_test, y_pred_balanced)
precision_bal = precision_score(y_test, y_pred_balanced)
recall_bal = recall_score(y_test, y_pred_balanced)
f1_bal = f1_score(y_test, y_pred_balanced)
roc_auc_bal = roc_auc_score(y_test, y_prob_balanced)
pr_auc_bal = average_precision_score(y_test, y_prob_balanced)

print("Balanced Logistic Regression Results")
print("=" * 45)

print(f"Accuracy : {accuracy_bal:.4f}")
print(f"Precision: {precision_bal:.4f}")
print(f"Recall   : {recall_bal:.4f}")
print(f"F1-Score : {f1_bal:.4f}")
print(f"ROC-AUC  : {roc_auc_bal:.4f}")
print(f"PR-AUC   : {pr_auc_bal:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_balanced,
    digits=4
))

  import RandomForestClassifier

In [ ]:
import time
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1 ## use all cpu cores
)

start_time = time.time()

random_forest.fit(
    X_train_processed,
    y_train
)

rf_training_time = time.time() - start_time

print("Random Forest training completed!")
print(f"Training time: {rf_training_time:.2f} seconds")

print("\nModel parameters:")
print("Number of trees:", random_forest.n_estimators)
print("Max depth:", random_forest.max_depth)
print("Class weight:", random_forest.class_weight)

 Random Forest predictions..... Evaluation metrics

In [ ]:
y_pred_rf = random_forest.predict(X_test_processed)
y_prob_rf = random_forest.predict_proba(X_test_processed)[:, 1]
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
pr_auc_rf = average_precision_score(y_test, y_prob_rf)

print("Baseline Random Forest Results")
print("=" * 45)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1-Score : {f1_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_rf:.4f}")
print(f"PR-AUC   : {pr_auc_rf:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_rf,
        digits=4
    )
)

Random Forest overfitting check

In [ ]:

y_train_pred_rf = random_forest.predict(X_train_processed)
y_train_prob_rf = random_forest.predict_proba(X_train_processed)[:, 1]


train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
train_f1_rf = f1_score(y_train, y_train_pred_rf)
train_roc_auc_rf = roc_auc_score(y_train, y_train_prob_rf)
print("Random Forest: Train vs Test")
print("=" * 45)

print(f"{'Metric':<15} {'Train':>10} {'Test':>10}")
print("-" * 37)

print(
    f"{'Accuracy':<15} "
    f"{train_accuracy_rf:>10.4f} "
    f"{accuracy_rf:>10.4f}"
)

print(
    f"{'F1-Score':<15} "
    f"{train_f1_rf:>10.4f} "
    f"{f1_rf:>10.4f}"
)

print(
    f"{'ROC-AUC':<15} "
    f"{train_roc_auc_rf:>10.4f} "
    f"{roc_auc_rf:>10.4f}"
)

print("\nPerformance gaps:")

print(
    f"Accuracy gap: "
    f"{train_accuracy_rf - accuracy_rf:.4f}"
)

print(
    f"F1 gap: "
    f"{train_f1_rf - f1_rf:.4f}"
)

print(
    f"ROC-AUC gap: "
    f"{train_roc_auc_rf - roc_auc_rf:.4f}"
)

In [ ]:
 import time

rf_regularized = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

rf_regularized.fit(
    X_train_processed,
    y_train
)

rf_reg_training_time = time.time() - start_time

print("Regularized Random Forest training completed!")
print(f"Training time: {rf_reg_training_time:.2f} seconds")

print("\nParameters:")
print("Trees:", rf_regularized.n_estimators)
print("Max depth:", rf_regularized.max_depth)
print("Min samples split:", rf_regularized.min_samples_split)
print("Min samples leaf:", rf_regularized.min_samples_leaf)
print("Max features:", rf_regularized.max_features)
print("Class weight:", rf_regularized.class_weight)

Test  AND Train predictions and Train metrics


In [ ]:
y_pred_rf_reg = rf_regularized.predict(X_test_processed)
y_prob_rf_reg = rf_regularized.predict_proba(X_test_processed)[:, 1]
y_train_pred_rf_reg = rf_regularized.predict(X_train_processed)
y_train_prob_rf_reg = rf_regularized.predict_proba(X_train_processed)[:, 1]

accuracy_rf_reg = accuracy_score(y_test, y_pred_rf_reg)
precision_rf_reg = precision_score(y_test, y_pred_rf_reg)
recall_rf_reg = recall_score(y_test, y_pred_rf_reg)
f1_rf_reg = f1_score(y_test, y_pred_rf_reg)
roc_auc_rf_reg = roc_auc_score(y_test, y_prob_rf_reg)
pr_auc_rf_reg = average_precision_score(y_test, y_prob_rf_reg)
train_accuracy_rf_reg = accuracy_score(y_train, y_train_pred_rf_reg)
train_f1_rf_reg = f1_score(y_train, y_train_pred_rf_reg)
train_roc_auc_rf_reg = roc_auc_score(y_train, y_train_prob_rf_reg)

print("Regularized Random Forest Results")
print("=" * 45)

print(f"Accuracy : {accuracy_rf_reg:.4f}")
print(f"Precision: {precision_rf_reg:.4f}")
print(f"Recall   : {recall_rf_reg:.4f}")
print(f"F1-Score : {f1_rf_reg:.4f}")
print(f"ROC-AUC  : {roc_auc_rf_reg:.4f}")
print(f"PR-AUC   : {pr_auc_rf_reg:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_reg))

print("\nTrain vs Test:")
print(f"{'Metric':<15} {'Train':>10} {'Test':>10}")
print("-" * 37)
print(f"{'Accuracy':<15} {train_accuracy_rf_reg:>10.4f} {accuracy_rf_reg:>10.4f}")
print(f"{'F1-Score':<15} {train_f1_rf_reg:>10.4f} {f1_rf_reg:>10.4f}")
print(f"{'ROC-AUC':<15} {train_roc_auc_rf_reg:>10.4f} {roc_auc_rf_reg:>10.4f}")

print("\nPerformance gaps:")
print(f"Accuracy gap: {train_accuracy_rf_reg - accuracy_rf_reg:.4f}")
print(f"F1 gap: {train_f1_rf_reg - f1_rf_reg:.4f}")
print(f"ROC-AUC gap: {train_roc_auc_rf_reg - roc_auc_rf_reg:.4f}")

Balanced randomforestclassifier

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

rf_balanced.fit(
    X_train_processed,
    y_train
)

rf_bal_training_time = time.time() - start_time

print("Balanced Regularized Random Forest trained successfully!")
print(f"Training time: {rf_bal_training_time:.2f} seconds")

print("\nParameters:")
print("Trees:", rf_balanced.n_estimators)
print("Max depth:", rf_balanced.max_depth)
print("Min samples split:", rf_balanced.min_samples_split)
print("Min samples leaf:", rf_balanced.min_samples_leaf)
print("Max features:", rf_balanced.max_features)
print("Class weight:", rf_balanced.class_weight)

train, test and metrices

In [ ]:
y_pred_rf_bal = rf_balanced.predict(X_test_processed)
y_prob_rf_bal = rf_balanced.predict_proba(X_test_processed)[:, 1]
y_train_pred_rf_bal = rf_balanced.predict(X_train_processed)
y_train_prob_rf_bal = rf_balanced.predict_proba(X_train_processed)[:, 1]
accuracy_rf_bal = accuracy_score(y_test, y_pred_rf_bal)
precision_rf_bal = precision_score(y_test, y_pred_rf_bal)
recall_rf_bal = recall_score(y_test, y_pred_rf_bal)
f1_rf_bal = f1_score(y_test, y_pred_rf_bal)
roc_auc_rf_bal = roc_auc_score(y_test, y_prob_rf_bal)
pr_auc_rf_bal = average_precision_score(y_test, y_prob_rf_bal)
train_accuracy_rf_bal = accuracy_score(y_train, y_train_pred_rf_bal)
train_f1_rf_bal = f1_score(y_train, y_train_pred_rf_bal)
train_roc_auc_rf_bal = roc_auc_score(y_train, y_train_prob_rf_bal)

print("Balanced Regularized Random Forest Results")
print("=" * 50)

print(f"Accuracy : {accuracy_rf_bal:.4f}")
print(f"Precision: {precision_rf_bal:.4f}")
print(f"Recall   : {recall_rf_bal:.4f}")
print(f"F1-Score : {f1_rf_bal:.4f}")
print(f"ROC-AUC  : {roc_auc_rf_bal:.4f}")
print(f"PR-AUC   : {pr_auc_rf_bal:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_bal))

print("\nTrain vs Test:")
print(f"{'Metric':<15} {'Train':>10} {'Test':>10}")
print("-" * 37)

print(
    f"{'Accuracy':<15} "
    f"{train_accuracy_rf_bal:>10.4f} "
    f"{accuracy_rf_bal:>10.4f}"
)

print(
    f"{'F1-Score':<15} "
    f"{train_f1_rf_bal:>10.4f} "
    f"{f1_rf_bal:>10.4f}"
)

print(
    f"{'ROC-AUC':<15} "
    f"{train_roc_auc_rf_bal:>10.4f} "
    f"{roc_auc_rf_bal:>10.4f}"
)

print("\nPerformance gaps:")
print(
    f"Accuracy gap: "
    f"{train_accuracy_rf_bal - accuracy_rf_bal:.4f}"
)

print(
    f"F1 gap: "
    f"{train_f1_rf_bal - f1_rf_bal:.4f}"
)

print(
    f"ROC-AUC gap: "
    f"{train_roc_auc_rf_bal - roc_auc_rf_bal:.4f}"
)

comparison table of all alogorithyms that we applied   **Logistic Regression,
        Balanced Logistic Regression,
        Random Forest,
        Regularized Random Forest,
        Balanced Regularized Random Forest**

In [ ]:

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Balanced Logistic Regression",
        "Random Forest",
        "Regularized Random Forest",
        "Balanced Regularized Random Forest"
    ],

    "Accuracy": [
        accuracy,
        accuracy_bal,
        accuracy_rf,
        accuracy_rf_reg,
        accuracy_rf_bal
    ],

    "Precision": [
        precision,
        precision_bal,
        precision_rf,
        precision_rf_reg,
        precision_rf_bal
    ],

    "Recall": [
        recall,
        recall_bal,
        recall_rf,
        recall_rf_reg,
        recall_rf_bal
    ],

    "F1": [
        f1,
        f1_bal,
        f1_rf,
        f1_rf_reg,
        f1_rf_bal
    ],

    "ROC_AUC": [
        roc_auc,
        roc_auc_bal,
        roc_auc_rf,
        roc_auc_rf_reg,
        roc_auc_rf_bal
    ],

    "PR_AUC": [
        pr_auc,
        pr_auc_bal,
        pr_auc_rf,
        pr_auc_rf_reg,
        pr_auc_rf_bal
    ]
})

model_comparison = model_comparison.round(4)

print(model_comparison.to_string(index=False))

CROSS VALIDATION.........

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
oof_prob_rf_bal = cross_val_predict(
    rf_balanced,
    X_train_processed,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("Cross-validation completed.")
print("OOF probability shape:", oof_prob_rf_bal.shape)
print("Minimum probability:", oof_prob_rf_bal.min())
print("Maximum probability:", oof_prob_rf_bal.max())
print("Mean probability:", oof_prob_rf_bal.mean())

print(
    "OOF ROC-AUC:",
    round(roc_auc_score(y_train, oof_prob_rf_bal), 4)
)

print(
    "OOF PR-AUC:",
    round(average_precision_score(y_train, oof_prob_rf_bal), 4)
)

Find Best Threshold Using OOF Data

In [ ]:
thresholds = np.arange(0.20, 0.71, 0.02)

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (
        oof_prob_rf_bal >= threshold
    ).astype(int)

    precision_t = precision_score(
        y_train,
        y_pred_threshold,
        zero_division=0
    )

    recall_t = recall_score(
        y_train,
        y_pred_threshold
    )

    f1_t = f1_score(
        y_train,
        y_pred_threshold
    )

    threshold_results.append([
        threshold,
        precision_t,
        recall_t,
        f1_t
    ])


threshold_df = pd.DataFrame(
    threshold_results,
    columns=[
        "Threshold",
        "Precision",
        "Recall",
        "F1"
    ]
)

threshold_df = threshold_df.round(4)

print(threshold_df.to_string(index=False))

best_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

print("\nBest Threshold based on OOF F1")
print("=" * 40)

print("Threshold:", best_row["Threshold"])
print("Precision:", best_row["Precision"])
print("Recall   :", best_row["Recall"])
print("F1-Score :", best_row["F1"])

Lock threshold selected using training OOF predictions

In [ ]:

best_threshold = 0.52
y_pred_rf_tuned = (
    y_prob_rf_bal >= best_threshold
).astype(int)

accuracy_tuned = accuracy_score(y_test, y_pred_rf_tuned)
precision_tuned = precision_score(y_test, y_pred_rf_tuned)
recall_tuned = recall_score(y_test, y_pred_rf_tuned)
f1_tuned = f1_score(y_test, y_pred_rf_tuned)

print("Balanced Random Forest — Threshold 0.52")
print("=" * 48)

print(f"Accuracy : {accuracy_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall   : {recall_tuned:.4f}")
print(f"F1-Score : {f1_tuned:.4f}")
print(f"ROC-AUC  : {roc_auc_rf_bal:.4f}")
print(f"PR-AUC   : {pr_auc_rf_bal:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_tuned))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_rf_tuned,
        digits=4
    )
)

Define Random Forest Search Space

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_param_grid = {
    "n_estimators": [200, 300, 400, 500],

    "max_depth": [
        6, 8, 10, 12, 15, None
    ],

    "min_samples_split": [
        2, 5, 10, 20
    ],

    "min_samples_leaf": [
        1, 2, 5, 10
    ],

    "max_features": [
        "sqrt", "log2", 0.5
    ],

    "class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}

print("Random Forest parameter space created.")
print("\nParameters to tune:")

for parameter, values in rf_param_grid.items():
    print(f"{parameter}: {values}")

Configure RandomizedSearchCV

In [ ]:
cv_search = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring="f1",
    cv=cv_search,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("RandomizedSearchCV configured successfully!")

print("\nConfiguration:")
print("Iterations:", rf_search.n_iter)
print("CV folds:", rf_search.cv.n_splits)
print("Scoring metric:", rf_search.scoring)
print("Total model fits:", rf_search.n_iter * rf_search.cv.n_splits)

Run RandomizedSearchCV

In [ ]:
import time

start_time = time.time()

rf_search.fit(
    X_train_processed,
    y_train
)

search_time = time.time() - start_time

print("\nRandomized Search completed!")
print(f"Total search time: {search_time / 60:.2f} minutes")

print("\nBest CV F1-Score:")
print(round(rf_search.best_score_, 4))

print("\nBest Parameters:")
for parameter, value in rf_search.best_params_.items():
    print(f"{parameter}: {value}")

test pridictions and metrices

In [ ]:
best_rf = rf_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test_processed)
y_prob_best_rf = best_rf.predict_proba(X_test_processed)[:, 1]
accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)
precision_best_rf = precision_score(y_test, y_pred_best_rf)
recall_best_rf = recall_score(y_test, y_pred_best_rf)
f1_best_rf = f1_score(y_test, y_pred_best_rf)
roc_auc_best_rf = roc_auc_score(y_test, y_prob_best_rf)
pr_auc_best_rf = average_precision_score(y_test, y_prob_best_rf)

print("Tuned Random Forest — Final Test Results")
print("=" * 50)
print(f"Accuracy : {accuracy_best_rf:.4f}")
print(f"Precision: {precision_best_rf:.4f}")
print(f"Recall   : {recall_best_rf:.4f}")
print(f"F1-Score : {f1_best_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_best_rf:.4f}")
print(f"PR-AUC   : {pr_auc_best_rf:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_rf))
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_best_rf,
    digits=4
))

Random Forest Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_rf.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 Most Important Features")
print("=" * 50)

print(
    feature_importance
    .head(15)
    .to_string(index=False)
)

Create monthly credit utilization ratios

In [ ]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()
for i in range(1, 7):

    X_train_fe[f"UTILIZATION_{i}"] = (
        X_train_fe[f"BILL_AMT{i}"] /
        X_train_fe["LIMIT_BAL"]
    )

    X_test_fe[f"UTILIZATION_{i}"] = (
        X_test_fe[f"BILL_AMT{i}"] /
        X_test_fe["LIMIT_BAL"]
    )

print("Feature engineering completed.")

print("\nOriginal number of features:")
print(X_train.shape[1])

print("\nNew number of features:")
print(X_train_fe.shape[1])

print("\nNew utilization features:")
print([
    f"UTILIZATION_{i}"
    for i in range(1, 7)
])

print("\nSample:")
print(
    X_train_fe[
        [
            "LIMIT_BAL",
            "BILL_AMT1",
            "UTILIZATION_1"
        ]
    ].head()
)

Inspect Utilization Features

In [ ]:
utilization_cols = [
    f"UTILIZATION_{i}"
    for i in range(1, 7)
]

utilization_summary = X_train_fe[
    utilization_cols
].describe().T

utilization_summary["skewness"] = X_train_fe[
    utilization_cols
].skew()

print("Utilization Feature Summary")
print("=" * 70)

print(
    utilization_summary[
        [
            "min",
            "mean",
            "50%",
            "max",
            "skewness"
        ]
    ].round(4)
)

print("\nCounts above 100% utilization:")

for col in utilization_cols:
    count = (X_train_fe[col] > 1).sum()
    percentage = count / len(X_train_fe) * 100

    print(
        f"{col}: {count} "
        f"({percentage:.2f}%)"
    )

Average & Maximum Utilization

In [ ]:

X_train_fe["AVG_UTILIZATION"] = (
    X_train_fe[utilization_cols].mean(axis=1)
)

X_test_fe["AVG_UTILIZATION"] = (
    X_test_fe[utilization_cols].mean(axis=1)
)

X_train_fe["MAX_UTILIZATION"] = (
    X_train_fe[utilization_cols].max(axis=1)
)

X_test_fe["MAX_UTILIZATION"] = (
    X_test_fe[utilization_cols].max(axis=1)
)

print("Aggregate utilization features created.")

print("\nTraining shape:")
print(X_train_fe.shape)

print("\nAVG_UTILIZATION summary:")
print(
    X_train_fe["AVG_UTILIZATION"]
    .describe()
    .round(4)
)

print("\nMAX_UTILIZATION summary:")
print(
    X_train_fe["MAX_UTILIZATION"]
    .describe()
    .round(4)
)

 Average repayment status and Worst repayment delay
 with  Number of months with a positive delay status



In [ ]:
pay_status_cols = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

X_train_fe["AVG_DELAY"] = (
    X_train_fe[pay_status_cols].mean(axis=1)
)

X_test_fe["AVG_DELAY"] = (
    X_test_fe[pay_status_cols].mean(axis=1)
)

X_train_fe["MAX_DELAY"] = (
    X_train_fe[pay_status_cols].max(axis=1)
)

X_test_fe["MAX_DELAY"] = (
    X_test_fe[pay_status_cols].max(axis=1)
)

X_train_fe["DELAY_MONTHS"] = (
    X_train_fe[pay_status_cols] > 0
).sum(axis=1)

X_test_fe["DELAY_MONTHS"] = (
    X_test_fe[pay_status_cols] > 0
).sum(axis=1)

print("Repayment behavior features created.")

print("\nTraining shape:")
print(X_train_fe.shape)

print("\nNew features summary:")
print(
    X_train_fe[
        ["AVG_DELAY", "MAX_DELAY", "DELAY_MONTHS"]
    ].describe().round(3)
)

Engineered Features vs Default

In [ ]:
engineered_features = [
    "AVG_UTILIZATION",
    "MAX_UTILIZATION",
    "AVG_DELAY",
    "MAX_DELAY",
    "DELAY_MONTHS"
]

feature_target_analysis = []

for feature in engineered_features:

    temp = pd.DataFrame({
        "feature": X_train_fe[feature],
        "target": y_train.values
    })

    correlation = temp["feature"].corr(temp["target"])

    non_default_mean = temp.loc[
        temp["target"] == 0,
        "feature"
    ].mean()

    default_mean = temp.loc[
        temp["target"] == 1,
        "feature"
    ].mean()

    feature_target_analysis.append([
        feature,
        correlation,
        non_default_mean,
        default_mean
    ])

feature_target_df = pd.DataFrame(
    feature_target_analysis,
    columns=[
        "Feature",
        "Correlation_with_Default",
        "Mean_Non_Default",
        "Mean_Default"
    ]
)

print("Engineered Features vs Default")
print("=" * 70)

print(
    feature_target_df
    .round(4)
    .to_string(index=False)
)

Define Feature-Engineered Preprocessor

In [ ]:

categorical_features_fe = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]
numerical_features_fe = [
    col for col in X_train_fe.columns
    if col not in categorical_features_fe
]
preprocessor_fe = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features_fe
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features_fe
        )
    ]
)

print("Feature-engineered preprocessor created.")

print("\nNumerical features:")
print(len(numerical_features_fe))

print("\nCategorical features:")
print(len(categorical_features_fe))

print("\nTotal input features:")
print(
    len(numerical_features_fe)
    + len(categorical_features_fe)
)

Transform Feature-Engineered Data

In [ ]:

X_train_fe_processed = preprocessor_fe.fit_transform(
    X_train_fe
)
X_test_fe_processed = preprocessor_fe.transform(
    X_test_fe
)

print("Feature-engineered preprocessing completed.")

print("\nBefore preprocessing:")
print("X_train_fe:", X_train_fe.shape)
print("X_test_fe :", X_test_fe.shape)

print("\nAfter preprocessing:")
print(
    "X_train_fe_processed:",
    X_train_fe_processed.shape
)

print(
    "X_test_fe_processed :",
    X_test_fe_processed.shape
)

print("\nMissing values:")
print(
    "Train:",
    np.isnan(X_train_fe_processed).sum()
)

print(
    "Test :",
    np.isnan(X_test_fe_processed).sum()
)

print("\nInfinite values:")
print(
    "Train:",
    np.isinf(X_train_fe_processed).sum()
)

print(
    "Test :",
    np.isinf(X_test_fe_processed).sum()
)

Train RF on Engineered Features

In [ ]:
import time

rf_feature_engineered = RandomForestClassifier(
    n_estimators=500,
    min_samples_split=5,
    min_samples_leaf=5,
    max_features="sqrt",
    max_depth=12,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

rf_feature_engineered.fit(
    X_train_fe_processed,
    y_train
)

fe_training_time = time.time() - start_time

print("Feature-Engineered Random Forest trained successfully!")
print(f"Training time: {fe_training_time:.2f} seconds")

print("\nParameters:")
print("Trees:", rf_feature_engineered.n_estimators)
print("Max depth:", rf_feature_engineered.max_depth)
print("Min samples split:", rf_feature_engineered.min_samples_split)
print("Min samples leaf:", rf_feature_engineered.min_samples_leaf)
print("Max features:", rf_feature_engineered.max_features)
print("Class weight:", rf_feature_engineered.class_weight)

In [ ]:
cv_fe = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results_fe = cross_validate(
    rf_feature_engineered,
    X_train_fe_processed,
    y_train,
    cv=cv_fe,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision"
    ],
    n_jobs=-1
)

print("Feature-Engineered Random Forest — 5-Fold CV")
print("=" * 55)

print(
    f"Accuracy : "
    f"{cv_results_fe['test_accuracy'].mean():.4f}"
)

print(
    f"Precision: "
    f"{cv_results_fe['test_precision'].mean():.4f}"
)

print(
    f"Recall   : "
    f"{cv_results_fe['test_recall'].mean():.4f}"
)

print(
    f"F1-Score : "
    f"{cv_results_fe['test_f1'].mean():.4f}"
)

print(
    f"ROC-AUC  : "
    f"{cv_results_fe['test_roc_auc'].mean():.4f}"
)

print(
    f"PR-AUC   : "
    f"{cv_results_fe['test_average_precision'].mean():.4f}"
)

Engineered Feature Importance

In [ ]:

feature_names_fe = (
    preprocessor_fe.get_feature_names_out()
)
fe_importance = pd.DataFrame({
    "Feature": feature_names_fe,
    "Importance": rf_feature_engineered.feature_importances_
})

fe_importance = (
    fe_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 Features — Feature-Engineered Random Forest")
print("=" * 65)

print(
    fe_importance
    .head(20)
    .to_string(index=False)
)

Average Bill & Payment Features

In [ ]:
bill_cols = [
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6"
]

payment_cols = [
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

X_train_fe["AVG_BILL_AMT"] = (
    X_train_fe[bill_cols].mean(axis=1)
)

X_test_fe["AVG_BILL_AMT"] = (
    X_test_fe[bill_cols].mean(axis=1)
)

X_train_fe["AVG_PAY_AMT"] = (
    X_train_fe[payment_cols].mean(axis=1)
)

X_test_fe["AVG_PAY_AMT"] = (
    X_test_fe[payment_cols].mean(axis=1)
)

print("Average bill/payment features created.")

print("\nTraining shape:")
print(X_train_fe.shape)

print("\nNew feature summary:")
print(
    X_train_fe[
        ["AVG_BILL_AMT", "AVG_PAY_AMT"]
    ].describe().round(2)
)

Payment Behavior Ratios

In [ ]:

train_positive_bill = X_train_fe["AVG_BILL_AMT"].clip(lower=1)
test_positive_bill = X_test_fe["AVG_BILL_AMT"].clip(lower=1)

X_train_fe["PAY_TO_BILL_RATIO"] = (
    X_train_fe["AVG_PAY_AMT"] /
    train_positive_bill
)

X_test_fe["PAY_TO_BILL_RATIO"] = (
    X_test_fe["AVG_PAY_AMT"] /
    test_positive_bill
)
X_train_fe["PAY_TO_LIMIT_RATIO"] = (
    X_train_fe["AVG_PAY_AMT"] /
    X_train_fe["LIMIT_BAL"]
)

X_test_fe["PAY_TO_LIMIT_RATIO"] = (
    X_test_fe["AVG_PAY_AMT"] /
    X_test_fe["LIMIT_BAL"]
)

print("Payment behavior ratios created.")

print("\nTraining shape:")
print(X_train_fe.shape)

print("\nRatio summary:")
print(
    X_train_fe[
        [
            "PAY_TO_BILL_RATIO",
            "PAY_TO_LIMIT_RATIO"
        ]
    ].describe().round(4)
)

Fix PAY_TO_BILL_RATIO and Replace PAY_TO_BILL_RATIO with a safer version

In [ ]:

X_train_fe["PAY_TO_BILL_RATIO"] = np.where(
    X_train_fe["AVG_BILL_AMT"] > 0,
    X_train_fe["AVG_PAY_AMT"] / X_train_fe["AVG_BILL_AMT"],
    0
)

X_test_fe["PAY_TO_BILL_RATIO"] = np.where(
    X_test_fe["AVG_BILL_AMT"] > 0,
    X_test_fe["AVG_PAY_AMT"] / X_test_fe["AVG_BILL_AMT"],
    0
)

print("PAY_TO_BILL_RATIO corrected.")

print("\nUpdated summary:")
print(
    X_train_fe[
        ["PAY_TO_BILL_RATIO", "PAY_TO_LIMIT_RATIO"]
    ].describe().round(4)
)

print("\nExtreme PAY_TO_BILL_RATIO counts:")
print(
    "> 1 :",
    (X_train_fe["PAY_TO_BILL_RATIO"] > 1).sum()
)

print(
    "> 5 :",
    (X_train_fe["PAY_TO_BILL_RATIO"] > 5).sum()
)

print(
    "> 10:",
    (X_train_fe["PAY_TO_BILL_RATIO"] > 10).sum()
)

Check New Features vs Default

In [ ]:
new_features = [
    "AVG_BILL_AMT",
    "AVG_PAY_AMT",
    "PAY_TO_BILL_RATIO",
    "PAY_TO_LIMIT_RATIO"
]

feature_target_check = pd.DataFrame()

for feature in new_features:
    feature_target_check.loc[feature, "Correlation"] = (
        X_train_fe[feature].corr(y_train)
    )

    feature_target_check.loc[feature, "Mean_No_Default"] = (
        X_train_fe.loc[y_train == 0, feature].mean()
    )

    feature_target_check.loc[feature, "Mean_Default"] = (
        X_train_fe.loc[y_train == 1, feature].mean()
    )

print("New Features vs Default")
print("=" * 65)

print(
    feature_target_check.round(4)
)

Rebuild Preprocessor for 38 Features

In [ ]:
categorical_features_fe = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numerical_features_fe = [
    col for col in X_train_fe.columns
    if col not in categorical_features_fe
]

preprocessor_fe_v2 = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features_fe
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features_fe
        )
    ]
)

print("Updated feature-engineered preprocessor created.")

print("\nNumerical features:",
      len(numerical_features_fe))

print("Categorical features:",
      len(categorical_features_fe))

print("Total raw features:",
      X_train_fe.shape[1])

Transform Updated Feature Set

In [ ]:

X_train_fe_v2_processed = preprocessor_fe_v2.fit_transform(
    X_train_fe
)

X_test_fe_v2_processed = preprocessor_fe_v2.transform(
    X_test_fe
)

print("Updated feature set preprocessing completed.")

print("\nBefore preprocessing:")
print("Train:", X_train_fe.shape)
print("Test :", X_test_fe.shape)

print("\nAfter preprocessing:")
print("Train:", X_train_fe_v2_processed.shape)
print("Test :", X_test_fe_v2_processed.shape)

print("\nMissing values:")
print(
    "Train:",
    np.isnan(X_train_fe_v2_processed).sum()
)
print(
    "Test :",
    np.isnan(X_test_fe_v2_processed).sum()
)

print("\nInfinite values:")
print(
    "Train:",
    np.isinf(X_train_fe_v2_processed).sum()
)
print(
    "Test :",
    np.isinf(X_test_fe_v2_processed).sum()
)

CV on Updated Feature Set

In [ ]:
rf_fe_v2 = RandomForestClassifier(
    n_estimators=500,
    min_samples_split=5,
    min_samples_leaf=5,
    max_features="sqrt",
    max_depth=12,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

cv_results_fe_v2 = cross_validate(
    rf_fe_v2,
    X_train_fe_v2_processed,
    y_train,
    cv=cv_fe,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision"
    ],
    n_jobs=-1
)

print("Updated Feature-Engineered RF — 5-Fold CV")
print("=" * 55)

print(
    f"Accuracy : "
    f"{cv_results_fe_v2['test_accuracy'].mean():.4f}"
)

print(
    f"Precision: "
    f"{cv_results_fe_v2['test_precision'].mean():.4f}"
)

print(
    f"Recall   : "
    f"{cv_results_fe_v2['test_recall'].mean():.4f}"
)

print(
    f"F1-Score : "
    f"{cv_results_fe_v2['test_f1'].mean():.4f}"
)

print(
    f"ROC-AUC  : "
    f"{cv_results_fe_v2['test_roc_auc'].mean():.4f}"
)

print(
    f"PR-AUC   : "
    f"{cv_results_fe_v2['test_average_precision'].mean():.4f}"
)

CV Stability

In [ ]:
metrics = {
    "Accuracy": "test_accuracy",
    "Precision": "test_precision",
    "Recall": "test_recall",
    "F1": "test_f1",
    "ROC-AUC": "test_roc_auc",
    "PR-AUC": "test_average_precision"
}

print("Updated Feature-Engineered RF — CV Stability")
print("=" * 60)

for metric_name, key in metrics.items():
    scores = cv_results_fe_v2[key]

    print(
        f"{metric_name:9s}: "
        f"{scores.mean():.4f} ± {scores.std():.4f} "
        f"| Folds: {np.round(scores, 4)}"
    )

Define Focused Search Space

In [ ]:
rf_fe_param_grid = {
    "n_estimators": [
        400,
        500,
        600
    ],

    "max_depth": [
        10,
        12,
        14
    ],

    "min_samples_split": [
        3,
        5,
        8
    ],

    "min_samples_leaf": [
        3,
        5,
        7
    ],

    "max_features": [
        "sqrt",
        0.5
    ],

    "class_weight": [
        "balanced",
        "balanced_subsample"
    ]
}

print("Focused hyperparameter search space created.")

total_combinations = (
    len(rf_fe_param_grid["n_estimators"])
    * len(rf_fe_param_grid["max_depth"])
    * len(rf_fe_param_grid["min_samples_split"])
    * len(rf_fe_param_grid["min_samples_leaf"])
    * len(rf_fe_param_grid["max_features"])
    * len(rf_fe_param_grid["class_weight"])
)

print(
    "Total possible combinations:",
    total_combinations
)

Run Focused Hyperparameter Search

In [ ]:
rf_fe_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_fe_search = RandomizedSearchCV(
    estimator=rf_fe_base,
    param_distributions=rf_fe_param_grid,
    n_iter=15,
    scoring="f1",
    cv=cv_fe,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

import time
start_time = time.time()
rf_fe_search.fit(
    X_train_fe_v2_processed,
    y_train
)
search_time = time.time() - start_time
print("\nFocused tuning completed.")
print(f"Search time: {search_time / 60:.2f} minutes")
print("\nBest CV F1:")
print(round(rf_fe_search.best_score_, 4))
print("\nBest Parameters:")
for parameter, value in rf_fe_search.best_params_.items():
    print(f"{parameter}: {value}")

Extract Best Final Model

In [ ]:
best_rf_fe = rf_fe_search.best_estimator_

print("Best feature-engineered Random Forest selected.")

print("\nModel parameters:")
print("Trees            :", best_rf_fe.n_estimators)
print("Max depth        :", best_rf_fe.max_depth)
print("Min samples split:", best_rf_fe.min_samples_split)
print("Min samples leaf :", best_rf_fe.min_samples_leaf)
print("Max features     :", best_rf_fe.max_features)
print("Class weight     :", best_rf_fe.class_weight)

print("\nBest CV F1:")
print(round(rf_fe_search.best_score_, 4))

Final Test Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
y_test_pred_fe = best_rf_fe.predict(
    X_test_fe_v2_processed
)

y_test_prob_fe = best_rf_fe.predict_proba(
    X_test_fe_v2_processed
)[:, 1]
final_accuracy = accuracy_score(
    y_test, y_test_pred_fe
)

final_precision = precision_score(
    y_test, y_test_pred_fe
)

final_recall = recall_score(
    y_test, y_test_pred_fe
)

final_f1 = f1_score(
    y_test, y_test_pred_fe
)

final_roc_auc = roc_auc_score(
    y_test, y_test_prob_fe
)

final_pr_auc = average_precision_score(
    y_test, y_test_prob_fe
)

final_cm = confusion_matrix(
    y_test, y_test_pred_fe
)

print("Final Feature-Engineered Random Forest — Test Results")
print("=" * 60)

print(f"Accuracy : {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall   : {final_recall:.4f}")
print(f"F1-Score : {final_f1:.4f}")
print(f"ROC-AUC  : {final_roc_auc:.4f}")
print(f"PR-AUC   : {final_pr_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred_fe,
        digits=4
    )
)

print("Confusion Matrix:")
print(final_cm)

 CHECK DATA BEFORE STARTING XGBOOST

In [ ]:

required_variables = [
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "X_train_fe",
    "X_test_fe"
]

print("XGBoost Data Readiness Check")
print("=" * 45)

all_ready = True

for var in required_variables:
    exists = var in globals()
    print(f"{var:12} : {'READY' if exists else 'MISSING'}")

    if not exists:
        all_ready = False

print("=" * 45)

if all_ready:
    print("All data is ready. We can start XGBoost.")
    print("\nFeature-engineered shapes:")
    print("X_train_fe:", X_train_fe.shape)
    print("X_test_fe :", X_test_fe.shape)
    print("y_train   :", y_train.shape)
    print("y_test    :", y_test.shape)
else:
    print("Some required variables are missing.")

XGBOOST — BASELINE MODEL

In [ ]:

import time

xgb_baseline = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Training baseline XGBoost...")
print("=" * 50)

start_time = time.time()

xgb_baseline.fit(
    X_train_fe,
    y_train
)

training_time = time.time() - start_time

print("Training completed.")
print(f"Training time: {training_time:.2f} seconds")

XGBOOST — BASELINE EVALUATION

In [ ]:

xgb_test_pred = xgb_baseline.predict(X_test_fe)
xgb_test_prob = xgb_baseline.predict_proba(X_test_fe)[:, 1]

print("Baseline XGBoost — Test Results")
print("=" * 50)

print(f"Accuracy : {accuracy_score(y_test, xgb_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, xgb_test_pred):.4f}")
print(f"Recall   : {recall_score(y_test, xgb_test_pred):.4f}")
print(f"F1-Score : {f1_score(y_test, xgb_test_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, xgb_test_prob):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, xgb_test_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, xgb_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_test_pred))

CALCULATE CLASS IMBALANCE WEIGHT FOR XGBOOST

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Training Class Distribution")
print("=" * 45)
print("Non-Default (0):", negative_count)
print("Default (1)    :", positive_count)

print(
    "\nRecommended scale_pos_weight:",
    round(scale_pos_weight, 4)
)

XGBOOST — CLASS BALANCED MODEL

In [ ]:
xgb_balanced = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

print("Training class-balanced XGBoost...")
print("=" * 50)

start_time = time.time()

xgb_balanced.fit(
    X_train_fe,
    y_train
)

training_time = time.time() - start_time

print("Training completed.")
print(f"Training time: {training_time:.2f} seconds")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

XGBOOST — BALANCED MODEL EVALUATION

In [ ]:

xgb_balanced_pred = xgb_balanced.predict(X_test_fe)

xgb_balanced_prob = xgb_balanced.predict_proba(
    X_test_fe
)[:, 1]

print("Balanced XGBoost — Test Results")
print("=" * 50)

print(
    f"Accuracy : "
    f"{accuracy_score(y_test, xgb_balanced_pred):.4f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test, xgb_balanced_pred):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test, xgb_balanced_pred):.4f}"
)

print(
    f"F1-Score : "
    f"{f1_score(y_test, xgb_balanced_pred):.4f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_test, xgb_balanced_prob):.4f}"
)

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_test, xgb_balanced_prob):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        xgb_balanced_pred
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        xgb_balanced_pred
    )
)

XGBOOST — HYPERPARAMETER SEARCH SPACE

In [ ]:
xgb_param_grid = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.2],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 2, 5]
}

print("XGBoost hyperparameter search space ready.")
print("=" * 55)

for parameter, values in xgb_param_grid.items():
    print(f"{parameter:20}: {values}")

XGBOOST — RANDOMIZED SEARCH CV

In [ ]:

import time
xgb_tuning_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)
xgb_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_tuning_base,
    param_distributions=xgb_param_grid,
    n_iter=20,
    scoring="f1",
    cv=xgb_cv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

print("Starting XGBoost RandomizedSearchCV...")
print("=" * 55)
print("Candidates : 20")
print("CV folds   : 5")
print("Total fits : 100")
print("Scoring    : F1")
print()

start_time = time.time()

xgb_random_search.fit(
    X_train_fe,
    y_train
)

search_time = time.time() - start_time

print("\nXGBoost tuning completed.")
print(f"Search time: {search_time:.2f} seconds")

print("\nBest CV F1:")
print(f"{xgb_random_search.best_score_:.4f}")

print("\nBest Parameters:")
for parameter, value in xgb_random_search.best_params_.items():
    print(f"{parameter:20}: {value}")

TUNED XGBOOST — FINAL TEST EVALUATION

In [ ]:
best_xgb = xgb_random_search.best_estimator_
xgb_tuned_pred = best_xgb.predict(X_test_fe)
xgb_tuned_prob = best_xgb.predict_proba(
    X_test_fe
)[:, 1]

print("Tuned XGBoost — Final Test Results")
print("=" * 50)

print(
    f"Accuracy : "
    f"{accuracy_score(y_test, xgb_tuned_pred):.4f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test, xgb_tuned_pred):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test, xgb_tuned_pred):.4f}"
)

print(
    f"F1-Score : "
    f"{f1_score(y_test, xgb_tuned_pred):.4f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_test, xgb_tuned_prob):.4f}"
)

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_test, xgb_tuned_prob):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        xgb_tuned_pred
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        xgb_tuned_pred
    )
)

FINAL MODEL COMPARISON

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        0.8088,
        0.7768,
        accuracy_score(y_test, xgb_tuned_pred)
    ],

    "Precision": [
        0.6923,
        0.4962,
        precision_score(y_test, xgb_tuned_pred)
    ],

    "Recall": [
        0.2442,
        0.5863,
        recall_score(y_test, xgb_tuned_pred)
    ],

    "F1-Score": [
        0.3610,
        0.5375,
        f1_score(y_test, xgb_tuned_pred)
    ],

    "ROC-AUC": [
        0.7101,
        0.7777,
        roc_auc_score(y_test, xgb_tuned_prob)
    ],

    "PR-AUC": [
        0.4950,
        0.5548,
        average_precision_score(y_test, xgb_tuned_prob)
    ]
})

model_comparison = model_comparison.round(4)

print("Final Model Comparison")
print("=" * 75)

display(model_comparison)

SHAP — INSTALLATION / VERSION CHECK

In [ ]:
!pip -q install shap
import shap
print("SHAP version:", shap.__version__)
print("SHAP successfully loaded.")

UPLOAD FINAL MODEL FROM COMPUTER

In [ ]:
from google.colab import files
uploaded = files.upload()
print("\nUploaded files:")
for filename in uploaded.keys():
    print(filename)

LOAD FINAL RANDOM FOREST MODEL

In [ ]:

import joblib

model_path = "/content/credit_default_model_v1.joblib"

deployment_artifact = joblib.load(model_path)

final_pipeline = deployment_artifact["pipeline"]
threshold = deployment_artifact["threshold"]
final_preprocessor = final_pipeline.named_steps["preprocessor"]
final_rf_model = final_pipeline.named_steps["classifier"]

print("Final model restored successfully.")
print("=" * 55)

print("Model      :", deployment_artifact["model_name"])
print("Version    :", deployment_artifact["model_version"])
print("Threshold  :", threshold)
print("Classifier :", type(final_rf_model).__name__)

print("\nPipeline steps:")
print(final_pipeline.named_steps.keys())

 SHAP STEP 1 — PREPARE DATA FOR RANDOM FOREST

In [ ]:
X_train_shap = final_preprocessor.transform(X_train_fe)
X_test_shap = final_preprocessor.transform(X_test_fe)
shap_feature_names = final_preprocessor.get_feature_names_out()

print("SHAP Data Preparation")
print("=" * 55)

print("Original engineered features :", X_train_fe.shape[1])
print("Processed SHAP features      :", X_train_shap.shape[1])
print("Training samples             :", X_train_shap.shape[0])
print("Test samples                 :", X_test_shap.shape[0])

print("\nData type:")
print(type(X_test_shap))

print("\nFirst 15 processed feature names:")
for feature in shap_feature_names[:15]:
    print(feature)

SHAP STEP 2 — CALCULATE SHAP VALUES ON SAMPLE

In [ ]:
import shap
import numpy as np
X_shap_sample = X_test_shap[:500]

print("SHAP sample shape:", X_shap_sample.shape)

explainer = shap.TreeExplainer(final_rf_model)
shap_result = explainer(X_shap_sample)

print("\nSHAP calculation completed.")
print("=" * 55)

print("Result type       :", type(shap_result))
print("SHAP values shape :", np.asarray(shap_result.values).shape)
print("Base values shape :", np.asarray(shap_result.base_values).shape)

SHAP STEP 3 — GLOBAL FEATURE IMPORTANCE
CLASS 1 = DEFAULT

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
shap_values_default = shap_result.values[:, :, 1]
X_shap_df = pd.DataFrame(
    X_shap_sample,
    columns=shap_feature_names
)

print("Default-class SHAP shape:", shap_values_default.shape)
print("Feature matrix shape     :", X_shap_df.shape)
shap.summary_plot(
    shap_values_default,
    X_shap_df,
    plot_type="bar",
    max_display=15,
    show=False
)

plt.title("Top 15 Features Influencing Credit Default Prediction")
plt.tight_layout()
plt.show()

SHAP STEP 4 — BEESWARM PLOT
Shows direction + magnitude of feature impact

In [ ]:

shap.summary_plot(
    shap_values_default,
    X_shap_df,
    max_display=15,
    show=False
)

plt.title("SHAP Summary Plot — Default Risk")
plt.tight_layout()
plt.show()

 SHAP STEP 5 — SELECT ONE CUSTOMER FOR LOCAL EXPLANATION

In [ ]:

customer_index = 0
customer_probability = final_rf_model.predict_proba(
    X_shap_sample[customer_index].reshape(1, -1)
)[0, 1]

customer_prediction = int(customer_probability >= threshold)

print("Individual Customer Explanation")
print("=" * 55)

print("Customer index      :", customer_index)
print("Default probability :", round(customer_probability * 100, 2), "%")
print("Threshold           :", threshold)
print(
    "Prediction          :",
    "DEFAULT / HIGH RISK" if customer_prediction == 1
    else "NO DEFAULT / LOWER RISK"
)

print("\nActual test label    :", y_test.iloc[customer_index])

SHAP STEP 6 — LOCAL WATERFALL PLOT FOR ONE CUSTOMER and waterfall plot

In [ ]:
customer_index = 0
customer_explanation = shap.Explanation(
    values=shap_result.values[customer_index, :, 1],
    base_values=shap_result.base_values[customer_index, 1],
    data=X_shap_sample[customer_index],
    feature_names=shap_feature_names
)
shap.plots.waterfall(
    customer_explanation,
    max_display=15
)

SHAP STEP 7 — GLOBAL IMPORTANCE TABLE

In [ ]:

mean_abs_shap = np.abs(shap_values_default).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "Feature": shap_feature_names,
    "Mean_ABS_SHAP": mean_abs_shap
})
shap_importance_df["Feature"] = (
    shap_importance_df["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)
shap_importance_df = (
    shap_importance_df
    .sort_values("Mean_ABS_SHAP", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 Global SHAP Features")
print("=" * 55)

print(
    shap_importance_df.head(15).to_string(
        index=False
    )
)

 FAIRNESS STEP 1 — PERFORMANCE BY GENDER

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
test_probabilities = final_pipeline.predict_proba(X_test_fe)[:, 1]
test_predictions = (test_probabilities >= threshold).astype(int)
fairness_df = X_test_fe.copy()

fairness_df["Actual"] = y_test.values
fairness_df["Predicted"] = test_predictions
fairness_df["Default_Probability"] = test_probabilities

results = []

# Original UCI encoding:
# SEX = 1 -> Male
# SEX = 2 -> Female
gender_mapping = {
    1: "Male",
    2: "Female"
}

for gender_code, gender_name in gender_mapping.items():

    group = fairness_df[fairness_df["SEX"] == gender_code]

    results.append({
        "Gender": gender_name,
        "Samples": len(group),

        "Actual_Default_Rate":
            group["Actual"].mean(),

        "Predicted_Default_Rate":
            group["Predicted"].mean(),

        "Accuracy":
            accuracy_score(group["Actual"], group["Predicted"]),

        "Precision":
            precision_score(
                group["Actual"],
                group["Predicted"],
                zero_division=0
            ),

        "Recall":
            recall_score(
                group["Actual"],
                group["Predicted"],
                zero_division=0
            ),

        "F1":
            f1_score(
                group["Actual"],
                group["Predicted"],
                zero_division=0
            )
    })

gender_fairness = pd.DataFrame(results)

print("Gender Fairness Analysis")
print("=" * 75)

print(
    gender_fairness.round(4).to_string(index=False)
)

FAIRNESS STEP 2 — FORMAL GENDER FAIRNESS METRICS

In [ ]:
male = fairness_df[fairness_df["SEX"] == 1]
female = fairness_df[fairness_df["SEX"] == 2]
male_positive_rate = male["Predicted"].mean()
female_positive_rate = female["Predicted"].mean()
male_tpr = recall_score(
    male["Actual"],
    male["Predicted"],
    zero_division=0
)

female_tpr = recall_score(
    female["Actual"],
    female["Predicted"],
    zero_division=0
)
def false_positive_rate(group):
    actual = group["Actual"].values
    predicted = group["Predicted"].values

    fp = ((actual == 0) & (predicted == 1)).sum()
    tn = ((actual == 0) & (predicted == 0)).sum()

    return fp / (fp + tn)

male_fpr = false_positive_rate(male)
female_fpr = false_positive_rate(female)
demographic_parity_difference = (
    male_positive_rate - female_positive_rate
)

equal_opportunity_difference = (
    male_tpr - female_tpr
)

fpr_difference = (
    male_fpr - female_fpr
)

print("Formal Gender Fairness Metrics")
print("=" * 60)

print(
    f"Male predicted-default rate   : "
    f"{male_positive_rate:.4f}"
)

print(
    f"Female predicted-default rate : "
    f"{female_positive_rate:.4f}"
)

print(
    f"\nDemographic Parity Difference : "
    f"{demographic_parity_difference:.4f}"
)

print(
    f"Male TPR / Recall             : "
    f"{male_tpr:.4f}"
)

print(
    f"Female TPR / Recall           : "
    f"{female_tpr:.4f}"
)

print(
    f"Equal Opportunity Difference  : "
    f"{equal_opportunity_difference:.4f}"
)

print(
    f"\nMale False Positive Rate      : "
    f"{male_fpr:.4f}"
)

print(
    f"Female False Positive Rate    : "
    f"{female_fpr:.4f}"
)

print(
    f"FPR Difference                : "
    f"{fpr_difference:.4f}"
)

FAIRNESS STEP 3 — PERFORMANCE BY AGE GROUP

In [ ]:
 fairness_df["Age_Group"] = pd.cut(
    fairness_df["AGE"],
    bins=[0, 30, 40, 50, 60, float("inf")],
    labels=[
        "≤30",
        "31-40",
        "41-50",
        "51-60",
        "60+"
    ]
)

age_results = []

for age_group, group in fairness_df.groupby(
    "Age_Group",
    observed=True
):

    age_results.append({
        "Age_Group": str(age_group),
        "Samples": len(group),

        "Actual_Default_Rate":
            group["Actual"].mean(),

        "Predicted_Default_Rate":
            group["Predicted"].mean(),

        "Accuracy":
            accuracy_score(
                group["Actual"],
                group["Predicted"]
            ),

        "Precision":
            precision_score(
                group["Actual"],
                group["Predicted"],
                zero_division=0
            ),

        "Recall":
            recall_score(
                group["Actual"],
                group["Predicted"],
                zero_division=0
            ),

        "F1":
            f1_score(
                group["Actual"],
                group["Predicted"],
                zero_division=0
            ),

        "FPR":
            false_positive_rate(group)
    })

age_fairness = pd.DataFrame(age_results)

print("Age-Group Fairness Analysis")
print("=" * 90)

print(
    age_fairness.round(4).to_string(index=False)
)

FAIRNESS STEP 4 — FINAL FAIRNESS SUMMARY

In [ ]:
 fairness_summary = pd.DataFrame({
    "Metric": [
        "Gender Demographic Parity Difference",
        "Gender Equal Opportunity Difference",
        "Gender False Positive Rate Difference",
        "Lowest Age-Group Recall",
        "Highest Age-Group Recall",
        "Lowest Age-Group FPR",
        "Highest Age-Group FPR"
    ],

    "Value": [
        demographic_parity_difference,
        equal_opportunity_difference,
        fpr_difference,
        age_fairness["Recall"].min(),
        age_fairness["Recall"].max(),
        age_fairness["FPR"].min(),
        age_fairness["FPR"].max()
    ]
})

print("Final Fairness Audit Summary")
print("=" * 65)

print(
    fairness_summary.round(4).to_string(index=False)
)

print("\nImportant Note:")
print(
    "The 60+ age group contains only 58 test samples, "
    "so its fairness metrics should be interpreted cautiously."
)

REGULATORY REPORTING — STEP 1
CREATE MODEL GOVERNANCE SUMMARY

In [ ]:
 regulatory_summary = {
    "Model_Name": "Credit Default Random Forest",
    "Model_Version": "1.0",
    "Model_Type": "RandomForestClassifier",
    "Decision_Threshold": 0.50,

    "Dataset": "UCI Credit Card Default",
    "Dataset_Size": 30000,
    "Training_Samples": 24000,
    "Test_Samples": 6000,

    "Original_Features": 23,
    "Engineered_Features": 38,
    "Processed_Features": 41,

    "Test_Accuracy": 0.7768,
    "Test_Precision": 0.4962,
    "Test_Recall": 0.5863,
    "Test_F1": 0.5375,
    "Test_ROC_AUC": 0.7777,
    "Test_PR_AUC": 0.5548,

    "Explainability_Method": "SHAP TreeExplainer",

    "Top_SHAP_Feature": shap_importance_df.iloc[0]["Feature"],
    "Top_SHAP_Importance": shap_importance_df.iloc[0]["Mean_ABS_SHAP"],

    "Gender_DP_Difference": demographic_parity_difference,
    "Gender_EO_Difference": equal_opportunity_difference,
    "Gender_FPR_Difference": fpr_difference,

    "Fairness_Limitation":
        "The 60+ age group contains only 58 test samples.",

    "Intended_Use":
        "Educational credit default risk assessment and decision support.",

    "Important_Limitation":
        "Not validated for autonomous real-world lending decisions."
}

regulatory_df = pd.DataFrame(
    regulatory_summary.items(),
    columns=["Field", "Value"]
)

print("MODEL GOVERNANCE / REGULATORY SUMMARY")
print("=" * 75)

print(regulatory_df.to_string(index=False))

REGULATORY REPORTING — STEP 2
 SAVE MACHINE-READABLE GOVERNANCE REPORT

In [ ]:
import json
regulatory_report = {}

for key, value in regulatory_summary.items():

    if isinstance(value, (np.integer,)):
        value = int(value)

    elif isinstance(value, (np.floating,)):
        value = float(value)

    regulatory_report[key] = value
regulatory_report["Governance"] = {
    "Human_Oversight_Required": True,
    "Automatic_Loan_Approval": False,
    "Explainability_Available": True,
    "Fairness_Audit_Performed": True,
    "Threshold_Tuned_On_Test_Set": False
}
# Save report
report_path = "credit_risk_regulatory_report.json"

with open(report_path, "w") as file:
    json.dump(
        regulatory_report,
        file,
        indent=4
    )


print("Regulatory report saved successfully.")
print("=" * 60)
print("File:", report_path)

print("\nGovernance Controls:")
for key, value in regulatory_report["Governance"].items():
    print(f"{key}: {value}")

MLFLOW STEP 1 — INSTALL AND VERIFY

In [ ]:
 !pip -q install mlflow

import mlflow

print("MLflow successfully installed.")
print("=" * 50)
print("MLflow version:", mlflow.__version__)

MLFLOW STEP 2 — LOG FINAL RANDOM FOREST EXPERIMENT

In [ ]:
import mlflow
mlflow.set_experiment("Credit_Default_Prediction")

with mlflow.start_run(run_name="Final_Random_Forest_v1"):


    # MODEL PARAMETERS

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 3)
    mlflow.log_param("min_samples_leaf", 7)
    mlflow.log_param("max_features", "sqrt")
    mlflow.log_param("class_weight", "balanced_subsample")
    mlflow.log_param("threshold", 0.50)


    # TEST METRICS

    mlflow.log_metric("test_accuracy", 0.7768)
    mlflow.log_metric("test_precision", 0.4962)
    mlflow.log_metric("test_recall", 0.5863)
    mlflow.log_metric("test_f1", 0.5375)
    mlflow.log_metric("test_roc_auc", 0.7777)
    mlflow.log_metric("test_pr_auc", 0.5548)


    # FAIRNESS METRICS

    mlflow.log_metric(
        "gender_demographic_parity_difference",
        float(demographic_parity_difference)
    )

    mlflow.log_metric(
        "gender_equal_opportunity_difference",
        float(equal_opportunity_difference)
    )

    mlflow.log_metric(
        "gender_fpr_difference",
        float(fpr_difference)
    )

    # PROJECT INFORMATION

    mlflow.set_tag("model_version", "1.0")
    mlflow.set_tag("dataset", "UCI Credit Card Default")
    mlflow.set_tag("explainability", "SHAP TreeExplainer")
    mlflow.set_tag("purpose", "Credit Default Risk Assessment")

    run_id = mlflow.active_run().info.run_id

print("MLflow experiment logged successfully.")
print("=" * 60)
print("Experiment : Credit_Default_Prediction")
print("Run        : Final_Random_Forest_v1")
print("Run ID     :", run_id)

 MLFLOW STEP 3 — VERIFY SAVED EXPERIMENT

In [ ]:

import mlflow
saved_run = mlflow.get_run(run_id)

print("MLflow Run Verification")
print("=" * 65)

print("Run ID   :", saved_run.info.run_id)
print("Run Name :", saved_run.data.tags.get("mlflow.runName"))
print("Status   :", saved_run.info.status)

print("\nMODEL PARAMETERS")
print("-" * 65)

for key, value in saved_run.data.params.items():
    print(f"{key:25s}: {value}")

print("\nMODEL METRICS")
print("-" * 65)

for key, value in saved_run.data.metrics.items():
    print(f"{key:40s}: {value:.4f}")

MLFLOW STEP 4 — LOCATE MLFLOW STORAGE

In [ ]:
import os
import mlflow

print("Current directory :", os.getcwd())
print("Tracking URI      :", mlflow.get_tracking_uri())

print("\nMLflow-related files/folders in /content:")
print("=" * 65)

for item in os.listdir("/content"):
    if "mlflow" in item.lower() or "mlruns" in item.lower():
        path = os.path.join("/content", item)

        if os.path.isdir(path):
            print("FOLDER :", path)
        else:
            print(
                "FILE   :",
                path,
                f"({os.path.getsize(path) / 1024:.2f} KB)"
            )

MLFLOW STEP 5 — DOWNLOAD MLFLOW DATABASE BACKUP

In [ ]:

from google.colab import files
import os

mlflow_db_path = "/content/mlflow.db"

if os.path.exists(mlflow_db_path):

    size_kb = os.path.getsize(mlflow_db_path) / 1024

    print("MLflow database found.")
    print("=" * 55)
    print("File :", mlflow_db_path)
    print(f"Size : {size_kb:.2f} KB")

    print("\nStarting download...")

    files.download(mlflow_db_path)

else:
    print("MLflow database not found.")

Generate OOF Probabilities

In [ ]:
from sklearn.model_selection import cross_val_predict

oof_prob_fe = cross_val_predict(
    best_rf_fe,
    X_train_fe_v2_processed,
    y_train,
    cv=cv_fe,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF probabilities generated.")

print("\nShape:", oof_prob_fe.shape)
print("Minimum probability:", round(oof_prob_fe.min(), 4))
print("Maximum probability:", round(oof_prob_fe.max(), 4))
print("Mean probability   :", round(oof_prob_fe.mean(), 4))

print(
    "\nOOF ROC-AUC:",
    round(
        roc_auc_score(y_train, oof_prob_fe),
        4
    )
)

print(
    "OOF PR-AUC :",
    round(
        average_precision_score(y_train, oof_prob_fe),
        4
    )
)

Find Best F1 Threshold

In [ ]:
threshold_results = []

for threshold in np.arange(0.20, 0.71, 0.01):

    predictions = (
        oof_prob_fe >= threshold
    ).astype(int)

    precision = precision_score(
        y_train,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        predictions
    )

    f1 = f1_score(
        y_train,
        predictions
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_df_fe = pd.DataFrame(
    threshold_results
)

best_threshold_row = threshold_df_fe.loc[
    threshold_df_fe["F1"].idxmax()
]

best_threshold_fe = float(
    best_threshold_row["Threshold"]
)

print("Best OOF Threshold")
print("=" * 45)

print(
    f"Threshold: {best_threshold_fe:.2f}"
)

print(
    f"Precision: "
    f"{best_threshold_row['Precision']:.4f}"
)

print(
    f"Recall   : "
    f"{best_threshold_row['Recall']:.4f}"
)

print(
    f"F1-Score : "
    f"{best_threshold_row['F1']:.4f}"
)

print("\nTop 10 thresholds by F1:")

print(
    threshold_df_fe
    .sort_values("F1", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

Final Overfitting Check

In [ ]:

y_train_pred_final = best_rf_fe.predict(
    X_train_fe_v2_processed
)

y_train_prob_final = best_rf_fe.predict_proba(
    X_train_fe_v2_processed
)[:, 1]


train_accuracy_final = accuracy_score(
    y_train,
    y_train_pred_final
)

train_f1_final = f1_score(
    y_train,
    y_train_pred_final
)

train_roc_final = roc_auc_score(
    y_train,
    y_train_prob_final
)

train_pr_final = average_precision_score(
    y_train,
    y_train_prob_final
)

print("Final Model — Train vs Test")
print("=" * 55)

print(
    f"{'Metric':<12}"
    f"{'Train':>10}"
    f"{'Test':>10}"
    f"{'Gap':>10}"
)

print("-" * 42)

print(
    f"{'Accuracy':<12}"
    f"{train_accuracy_final:>10.4f}"
    f"{final_accuracy:>10.4f}"
    f"{train_accuracy_final-final_accuracy:>10.4f}"
)

print(
    f"{'F1':<12}"
    f"{train_f1_final:>10.4f}"
    f"{final_f1:>10.4f}"
    f"{train_f1_final-final_f1:>10.4f}"
)

print(
    f"{'ROC-AUC':<12}"
    f"{train_roc_final:>10.4f}"
    f"{final_roc_auc:>10.4f}"
    f"{train_roc_final-final_roc_auc:>10.4f}"
)

print(
    f"{'PR-AUC':<12}"
    f"{train_pr_final:>10.4f}"
    f"{final_pr_auc:>10.4f}"
    f"{train_pr_final-final_pr_auc:>10.4f}"
)

OOF vs Test Generalization Check

In [ ]:
oof_pred_final = (
    oof_prob_fe >= 0.50
).astype(int)
oof_accuracy = accuracy_score(
    y_train,
    oof_pred_final
)

oof_precision = precision_score(
    y_train,
    oof_pred_final
)

oof_recall = recall_score(
    y_train,
    oof_pred_final
)

oof_f1 = f1_score(
    y_train,
    oof_pred_final
)

oof_roc = roc_auc_score(
    y_train,
    oof_prob_fe
)

oof_pr = average_precision_score(
    y_train,
    oof_prob_fe
)

print("OOF vs Test — Generalization Check")
print("=" * 60)

print(
    f"{'Metric':<12}"
    f"{'OOF CV':>12}"
    f"{'Test':>12}"
    f"{'Gap':>12}"
)

print("-" * 48)

metrics_compare = [
    ("Accuracy", oof_accuracy, final_accuracy),
    ("Precision", oof_precision, final_precision),
    ("Recall", oof_recall, final_recall),
    ("F1", oof_f1, final_f1),
    ("ROC-AUC", oof_roc, final_roc_auc),
    ("PR-AUC", oof_pr, final_pr_auc)
]

for name, oof_value, test_value in metrics_compare:
    print(
        f"{name:<12}"
        f"{oof_value:>12.4f}"
        f"{test_value:>12.4f}"
        f"{oof_value-test_value:>12.4f}"
    )

Final Feature Importance

In [ ]:
final_feature_names = (
    preprocessor_fe_v2.get_feature_names_out()
)

final_feature_importance = pd.DataFrame({
    "Feature": final_feature_names,
    "Importance": best_rf_fe.feature_importances_
})

final_feature_importance = (
    final_feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Final Model — Top 20 Feature Importances")
print("=" * 65)

print(
    final_feature_importance
    .head(20)
    .round(6)
    .to_string(index=False)
)

In [ ]:
final_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_fe_v2
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=400,
                max_depth=10,
                min_samples_split=3,
                min_samples_leaf=7,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Final deployment pipeline created successfully.")

print("\nPipeline steps:")
for name, step in final_pipeline.steps:
    print(f"{name}: {step.__class__.__name__}")

print("\nClassification threshold: 0.50")

Train Final Deployment Pipeline

In [ ]:
import time

start_time = time.time()

final_pipeline.fit(
    X_train_fe,
    y_train
)

pipeline_training_time = time.time() - start_time

print("Final deployment pipeline trained successfully.")

print(
    f"Training time: "
    f"{pipeline_training_time:.2f} seconds"
)

print("\nPipeline fitted:")
print(
    hasattr(
        final_pipeline.named_steps["classifier"],
        "classes_"
    )
)

print(
    "\nClasses:",
    final_pipeline.named_steps[
        "classifier"
    ].classes_
)

Verify Deployment Pipeline

In [ ]:
pipeline_test_pred = final_pipeline.predict(
    X_test_fe
)

pipeline_test_prob = final_pipeline.predict_proba(
    X_test_fe
)[:, 1]

print("Deployment Pipeline — Verification")
print("=" * 55)

print(
    f"Accuracy : "
    f"{accuracy_score(y_test, pipeline_test_pred):.4f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test, pipeline_test_pred):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test, pipeline_test_pred):.4f}"
)

print(
    f"F1-Score : "
    f"{f1_score(y_test, pipeline_test_pred):.4f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_test, pipeline_test_prob):.4f}"
)

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_test, pipeline_test_prob):.4f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        pipeline_test_pred
    )
)

print(
    "\nPredictions identical to final model:",
    np.array_equal(
        pipeline_test_pred,
        y_test_pred_fe
    )
)

print(
    "Probabilities identical:",
    np.allclose(
        pipeline_test_prob,
        y_test_prob_fe
    )
)

Save Final Model

In [ ]:
import joblib

deployment_artifact = {
    "pipeline": final_pipeline,
    "threshold": 0.50,
    "model_name": "Credit Default Random Forest",
    "model_version": "1.0",
    "features": list(X_train_fe.columns),
    "target": "default.payment.next.month"
}

model_path = "credit_default_model_v1.joblib"

joblib.dump(
    deployment_artifact,
    model_path
)

print("Final deployment model saved successfully.")
print("File:", model_path)

import os

file_size_mb = os.path.getsize(model_path) / (1024 ** 2)

print(
    f"File size: {file_size_mb:.2f} MB"
)

In [ ]:
loaded_artifact = joblib.load(
    "credit_default_model_v1.joblib"
)

loaded_pipeline = loaded_artifact["pipeline"]
loaded_threshold = loaded_artifact["threshold"]

print("Saved model loaded successfully.")

print("\nModel metadata:")
print("Name      :", loaded_artifact["model_name"])
print("Version   :", loaded_artifact["model_version"])
print("Threshold :", loaded_threshold)
print("Target    :", loaded_artifact["target"])
print(
    "Features  :",
    len(loaded_artifact["features"])
)
loaded_prob = loaded_pipeline.predict_proba(
    X_test_fe
)[:, 1]
loaded_pred = (
    loaded_prob >= loaded_threshold
).astype(int)

print(
    "\nPredictions identical:",
    np.array_equal(
        loaded_pred,
        pipeline_test_pred
    )
)

print(
    "Probabilities identical:",
    np.allclose(
        loaded_prob,
        pipeline_test_prob
    )
)

Automatic Feature Engineering Function

In [ ]:
def add_credit_features(data):
    df_fe = data.copy()
    utilization_cols = []

    for i in range(1, 7):
        col = f"UTILIZATION_{i}"

        df_fe[col] = (
            df_fe[f"BILL_AMT{i}"]
            / df_fe["LIMIT_BAL"]
        )

        utilization_cols.append(col)
    df_fe["AVG_UTILIZATION"] = (
        df_fe[utilization_cols].mean(axis=1)
    )

    df_fe["MAX_UTILIZATION"] = (
        df_fe[utilization_cols].max(axis=1)
    )
    pay_status_cols = [
        "PAY_0",
        "PAY_2",
        "PAY_3",
        "PAY_4",
        "PAY_5",
        "PAY_6"
    ]

    df_fe["AVG_DELAY"] = (
        df_fe[pay_status_cols].mean(axis=1)
    )

    df_fe["MAX_DELAY"] = (
        df_fe[pay_status_cols].max(axis=1)
    )

    df_fe["DELAY_MONTHS"] = (
        (df_fe[pay_status_cols] > 0)
        .sum(axis=1)
    )
    bill_cols = [
        f"BILL_AMT{i}"
        for i in range(1, 7)
    ]

    payment_cols = [
        f"PAY_AMT{i}"
        for i in range(1, 7)
    ]

    df_fe["AVG_BILL_AMT"] = (
        df_fe[bill_cols].mean(axis=1)
    )

    df_fe["AVG_PAY_AMT"] = (
        df_fe[payment_cols].mean(axis=1)
    )
    df_fe["PAY_TO_BILL_RATIO"] = np.where(
        df_fe["AVG_BILL_AMT"] > 0,
        df_fe["AVG_PAY_AMT"]
        / df_fe["AVG_BILL_AMT"],
        0
    )

    df_fe["PAY_TO_LIMIT_RATIO"] = (
        df_fe["AVG_PAY_AMT"]
        / df_fe["LIMIT_BAL"]
    )

    return df_fe
X_train_fe_check = add_credit_features(
    X_train
)

print(
    "Original features:",
    X_train.shape[1]
)

print(
    "After feature engineering:",
    X_train_fe_check.shape[1]
)

print(
    "Expected features:",
    X_train_fe.shape[1]
)

print(
    "\nColumns identical:",
    list(X_train_fe_check.columns)
    == list(X_train_fe.columns)
)

print(
    "Values identical:",
    np.allclose(
        X_train_fe_check.values,
        X_train_fe.values,
        equal_nan=True
    )
)

In [ ]:
import json

original_input_features = list(X_train.columns)

with open(
    "input_features.json",
    "w"
) as f:
    json.dump(
        original_input_features,
        f,
        indent=4
    )

print("Input feature configuration saved.")

print(
    "\nNumber of original input features:",
    len(original_input_features)
)

print("\nRequired customer inputs:")

for i, feature in enumerate(
    original_input_features,
    start=1
):
    print(f"{i}. {feature}")

feature engineering.py

In [ ]:
feature_engineering_code = '''
import numpy as np


def add_credit_features(data):
    """
    Convert the 23 original credit features
    into the 38 features expected by the model.
    """

    df_fe = data.copy()

    # Monthly utilization
    utilization_cols = []

    for i in range(1, 7):
        col = f"UTILIZATION_{i}"

        df_fe[col] = (
            df_fe[f"BILL_AMT{i}"]
            / df_fe["LIMIT_BAL"]
        )

        utilization_cols.append(col)

    # Aggregate utilization
    df_fe["AVG_UTILIZATION"] = (
        df_fe[utilization_cols].mean(axis=1)
    )

    df_fe["MAX_UTILIZATION"] = (
        df_fe[utilization_cols].max(axis=1)
    )

    # Repayment behaviour
    pay_status_cols = [
        "PAY_0",
        "PAY_2",
        "PAY_3",
        "PAY_4",
        "PAY_5",
        "PAY_6"
    ]

    df_fe["AVG_DELAY"] = (
        df_fe[pay_status_cols].mean(axis=1)
    )

    df_fe["MAX_DELAY"] = (
        df_fe[pay_status_cols].max(axis=1)
    )

    df_fe["DELAY_MONTHS"] = (
        (df_fe[pay_status_cols] > 0)
        .sum(axis=1)
    )

    # Bill and payment averages
    bill_cols = [
        f"BILL_AMT{i}"
        for i in range(1, 7)
    ]

    payment_cols = [
        f"PAY_AMT{i}"
        for i in range(1, 7)
    ]

    df_fe["AVG_BILL_AMT"] = (
        df_fe[bill_cols].mean(axis=1)
    )

    df_fe["AVG_PAY_AMT"] = (
        df_fe[payment_cols].mean(axis=1)
    )

    # Payment ratios
    df_fe["PAY_TO_BILL_RATIO"] = np.where(
        df_fe["AVG_BILL_AMT"] > 0,
        df_fe["AVG_PAY_AMT"]
        / df_fe["AVG_BILL_AMT"],
        0
    )

    df_fe["PAY_TO_LIMIT_RATIO"] = (
        df_fe["AVG_PAY_AMT"]
        / df_fe["LIMIT_BAL"]
    )

    return df_fe
'''

with open(
    "feature_engineering.py",
    "w"
) as f:
    f.write(feature_engineering_code)

print(
    "feature_engineering.py created successfully."
)

DOWNLOAD FILES

In [ ]:
from google.colab import files

files.download(
    "credit_default_model_v1.joblib"
)

files.download(
    "input_features.json"
)

files.download(
    "feature_engineering.py"
)